In [2]:
import os  
from dotenv import load_dotenv 
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [6]:
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage
from langchain_groq import ChatGroq 

llm = ChatGroq(model_name = "llama-3.1-8b-instant",groq_api_key=groq_api_key)

In [ ]:
#TEXT Summarization technique  - for limited/small num of tokens

speech = """I have a dream that one day down in Alabama, with its vicious racists, with its governor having his lips dripping with the words of interposition and nullification – one day right there in Alabama little black boys and black girls will be able to join hands with little white boys and white girls as sisters and brothers.
I have a dream today.I have a dream that one day every valley shall be exalted, and every hill and mountain shall be made low, the rough places will be made plain, and the crooked places will be made straight, and the glory of the Lord shall be revealed and all flesh shall see it together.
This is our hope. This is the faith that I go back to the South with. With this faith we will be able to hew out of the mountain of despair a stone of hope. With this faith we will be able to transform the jangling discords of our nation into a beautiful symphony of brotherhood. With this faith we will be able to work together, to pray together, 
to struggle together, to go to jail together, to stand up for freedom together, knowing that we will be free one day.
This will be the day, this will be the day when all of God’s children will be able to sing with new meaning “My country ’tis of thee, sweet land of liberty, of thee I sing. Land where my father’s died, land of the Pilgrim’s pride, from every mountainside, let freedom ring!"""

In [12]:
chat_message=[
    SystemMessage(content="You are expert with experise in summarizing speeched"),
    HumanMessage(content=f"Please provide a short and concisse summary of the follow speech:\n Text:{speech}")
]

In [9]:
llm.get_num_tokens(speech)

311

In [14]:
result = llm.invoke(chat_message)
result.content

'Here is a concise summary of Martin Luther King Jr.\'s speech:\n\nMartin Luther King Jr. expresses his vision of a future where racial segregation and inequality are overcome. He dreams of a day when black and white children can join hands as equals. He emphasizes the importance of faith and unity, stating that together, they can overcome the challenges of the nation and achieve true freedom and equality. He concludes with the iconic phrase "Let freedom ring," symbolizing a future where all people can enjoy the promise of liberty and equality.'

In [18]:
##Another way - using prompt teplate /  llm chains

# old way 
# from langchain.chains import LLMChain
# from langchain import PromptTemplate

# 2026
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Define your prompt
prompt = ChatPromptTemplate.from_template("Summarize this speech: {speech}")

# 2. Create the chain using the pipe operator (|)
# This replaces LLMChain entirely
chain = prompt | llm | StrOutputParser()

# 3. Invoke the chain
response = chain.invoke({"speech": speech})
print(response)


This is a famous speech by Martin Luther King Jr., delivered in 1963. In it, he expresses his vision for a future where racial equality and harmony are achieved. He describes his dream of a day when:

- Black children in Alabama can join hands with white children as equals
- All people, regardless of their background, can live together in unity and brotherhood
- The nation's divisions are transformed into a symphony of unity
- People from different backgrounds can work, pray, and struggle together for freedom

He also refers to the Bible, using phrases like "the glory of the Lord shall be revealed" and "let freedom ring," to emphasize the moral and spiritual basis of his vision. The speech concludes with a powerful call to action, urging people to work towards this vision of a more just and equal society.


STUFF DOCUMENT CHAIN SUMMARIZATION  

data(if it fits llm window) => prompttemplate=>llm => final summary

In [ ]:
from langchain_community.document_loaders import PyPDFLoader 
from langchain_core.prompts import ChatPromptTemplate

loader = PyPDFLoader("apjspeech.pdf")
docs = loader.load_and_split()
docs

[Document(metadata={'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'moddate': 'D:20070730160943', 'title': 'Microsoft Word - Document1', 'author': 'Shri', 'source': 'apjspeech.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those livi ng abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I \nenjoyed every minute of my tenure enriched by the wonderful assoc iation from each one \nof you, hailing from different walks of life, be it politics, sci ence and technology, \nacademics, arts, literature, business, judiciary, administration, local bodies, farming, \nhome makers, special children, media and above all from the youth and st udent \ncommunity who are the future wealt

In [21]:
prompt = ChatPromptTemplate.from_template(
    """ Write a concise and short summary of the following speech,
Speech :{text} """ 
)

In [23]:
from langchain_classic.chains.summarize import load_summarize_chain

chain = load_summarize_chain(llm,chain_type="stuff",prompt=prompt,verbose=True)
resp = chain.invoke(docs)
resp



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Human:  Write a concise and short summary of the following speech,
Speech :A P J Abdul Kalam Departing speech 
 
 
Friends, I am delighted to address you all, in the country and those livi ng abroad, after 
working with you and completing five beautiful and eventful years in Rashtrapati 
Bhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I 
enjoyed every minute of my tenure enriched by the wonderful assoc iation from each one 
of you, hailing from different walks of life, be it politics, sci ence and technology, 
academics, arts, literature, business, judiciary, administration, local bodies, farming, 
home makers, special children, media and above all from the youth and st udent 
community who are the future wealth of our country. During my intera ction at 
Rashtrapati Bhavan in Delhi and at every state and union territor y as well as through my 
on

{'input_documents': [Document(metadata={'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'moddate': 'D:20070730160943', 'title': 'Microsoft Word - Document1', 'author': 'Shri', 'source': 'apjspeech.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those livi ng abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I \nenjoyed every minute of my tenure enriched by the wonderful assoc iation from each one \nof you, hailing from different walks of life, be it politics, sci ence and technology, \nacademics, arts, literature, business, judiciary, administration, local bodies, farming, \nhome makers, special children, media and above all from the youth and st udent \ncommunity who 

In [24]:
from langchain_core.output_parsers import StrOutputParser

# 1. Define your chain (Prompt -> LLM -> String Parser)
# This is much faster and clearer
summarize_chain = prompt | llm | StrOutputParser()

# 2. Extract text from your docs (APJ Speech)
# Join all page content into one big string to "stuff" it
text_content = "\n\n".join([doc.page_content for doc in docs])

# 3. Run it
# Note: Use 'text' because that's usually the variable in your prompt
resp = summarize_chain.invoke({"text": text_content})
print(resp)

Here's a concise and short summary of the speech:

President A.P.J. Abdul Kalam's departing speech emphasizes the importance of accelerating development, empowering villages, and mobilizing rural core competence for competitiveness. He highlights the need for a developed India by 2020, driven by the aspirations of the youth and the potential of the 540 million young Indians.

Kalam shares various experiences and initiatives that showcase the country's progress, including:

1. Empowering villages through the provision of physical, electronic, and knowledge connectivity.
2. Mobilizing rural core competence through PURA (Providing Urban Amenities in Rural Areas) complexes.
3. Fostering agricultural growth through the "Seed to Food" initiative.
4. Promoting innovation and entrepreneurship through partnerships with experts and institutions.
5. Addressing the impact of disasters through partnership and community participation.
6. Defending the nation through the bravery and dedication of the

MAP REDUCE SUMMARZN TECHNIQUE 

DATA => CHUNKS => CHUNK PROMPT TEMPLATE1 => LLM => SUMMARY => P-TEMPLATE2 => FINAL SUMMARY

In [26]:
docs

[Document(metadata={'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'moddate': 'D:20070730160943', 'title': 'Microsoft Word - Document1', 'author': 'Shri', 'source': 'apjspeech.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those livi ng abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I \nenjoyed every minute of my tenure enriched by the wonderful assoc iation from each one \nof you, hailing from different walks of life, be it politics, sci ence and technology, \nacademics, arts, literature, business, judiciary, administration, local bodies, farming, \nhome makers, special children, media and above all from the youth and st udent \ncommunity who are the future wealt

In [27]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000,chunk_overlap=200)
final_docs = text_splitter.split_documents(docs)
final_docs

[Document(metadata={'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'moddate': 'D:20070730160943', 'title': 'Microsoft Word - Document1', 'author': 'Shri', 'source': 'apjspeech.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those livi ng abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I \nenjoyed every minute of my tenure enriched by the wonderful assoc iation from each one \nof you, hailing from different walks of life, be it politics, sci ence and technology, \nacademics, arts, literature, business, judiciary, administration, local bodies, farming, \nhome makers, special children, media and above all from the youth and st udent \ncommunity who are the future wealt

In [28]:
len(final_docs)

13

In [29]:
chunk_template = """ 
Please summarize the below speech:
Speech:`{text}'
Summary:
"""

chunks_prompt = ChatPromptTemplate.from_template(
    template=chunk_template
)

In [30]:
final_template = """
Provide the final summary of the entire speech with these important points.
Add a Motivation Title,Start the precise summary with an introduction and provide the summary in number 
points for the speech.
Speech:{text}
"""

final_prompt = ChatPromptTemplate.from_template(
    template=final_template
)

In [32]:
from langchain_classic.chains.summarize import load_summarize_chain

chain = load_summarize_chain(llm,
                             chain_type="map_reduce",
                             map_prompt=chunks_prompt,
                             combine_prompt=final_prompt,
                             verbose=True)
resp = chain.invoke(final_docs)
resp



> Entering new MapReduceDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Human:  
Please summarize the below speech:
Speech:`A P J Abdul Kalam Departing speech 
 
 
Friends, I am delighted to address you all, in the country and those livi ng abroad, after 
working with you and completing five beautiful and eventful years in Rashtrapati 
Bhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I 
enjoyed every minute of my tenure enriched by the wonderful assoc iation from each one 
of you, hailing from different walks of life, be it politics, sci ence and technology, 
academics, arts, literature, business, judiciary, administration, local bodies, farming, 
home makers, special children, media and above all from the youth and st udent 
community who are the future wealth of our country. During my intera ction at 
Rashtrapati Bhavan in Delhi and at every state and union territor y as well as through my 
online interactions, 

{'input_documents': [Document(metadata={'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'moddate': 'D:20070730160943', 'title': 'Microsoft Word - Document1', 'author': 'Shri', 'source': 'apjspeech.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those livi ng abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I \nenjoyed every minute of my tenure enriched by the wonderful assoc iation from each one \nof you, hailing from different walks of life, be it politics, sci ence and technology, \nacademics, arts, literature, business, judiciary, administration, local bodies, farming, \nhome makers, special children, media and above all from the youth and st udent \ncommunity who 

In [34]:
# --- THE 2026 STANDARD FOR MAP-REDUCE ---

# 1. THE MAP STEP: Summarize each chunk
# 'map_chain' is now an LCEL pipe
map_chain = chunks_prompt | llm | StrOutputParser()

# 2. RUN MAP: Use .batch() for parallel processing
# This sends all chunks to Groq at once
inputs = [{"text": d.page_content} for d in final_docs]
summaries = map_chain.batch(inputs)

# 3. THE REDUCE STEP: Combine those summaries
reduce_chain = final_prompt | llm | StrOutputParser()

# 4. RUN REDUCE: Join the results and get the final output
combined_text = "\n\n".join(summaries)
final_result = reduce_chain.invoke({"text": combined_text})

print(final_result)

**Motivation Title:** Empowering India: A Vision for a Developed Nation

**Precise Summary with Important Points:**

This speech by Dr. A. P. J. Abdul Kalam, the former President of India, is a departing address where he expresses gratitude to the people of India and abroad for the opportunity to serve as the President. He shares 10 important messages that he wants to share with the nation during his tenure. Here are the key points:

1. **Accelerate Development**: Focus on accelerating development and catering to the aspirations of the youth.
2. **Empower Villages**: Empower villages and mobilize rural core competence for competitiveness.
3. **Agricultural Growth**: Focus on agricultural growth from seed to food.
4. **Partnership and Collaboration**: Overcome problems through partnership and collaboration.
5. **Courage in Calamities**: Demonstrate courage in combating calamities and crises.
6. **Connectivity**: Establish connectivity for societal transformation.
7. **Defending the Nati

REFIINE TEXT SUMMARIZN TECHNIQUE

In [35]:
chain = load_summarize_chain(llm,
                             chain_type="refine",
                             verbose=True)
resp = chain.invoke(final_docs)
resp



> Entering new RefineDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Write a concise summary of the following:


"A P J Abdul Kalam Departing speech 
 
 
Friends, I am delighted to address you all, in the country and those livi ng abroad, after 
working with you and completing five beautiful and eventful years in Rashtrapati 
Bhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I 
enjoyed every minute of my tenure enriched by the wonderful assoc iation from each one 
of you, hailing from different walks of life, be it politics, sci ence and technology, 
academics, arts, literature, business, judiciary, administration, local bodies, farming, 
home makers, special children, media and above all from the youth and st udent 
community who are the future wealth of our country. During my intera ction at 
Rashtrapati Bhavan in Delhi and at every state and union territor y as well as through my 
online interactions, I have man

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kkrkbwkxfmnb24xc48an8tmy` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 3298, Requested 2781. Please try again in 790ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}